# 核技巧为什么不用显式升维？

**面试回答主线：**当算法只依赖样本内积时，可把 φ(x)ᵀφ(z) 替换为核函数 K(x,z)，从而在隐式高维空间表达非线性边界。核技巧省去显式构造交叉特征，却仍需计算样本两两相似度。本实验以 XOR 型优惠券审核为例，手写 RBF 核矩阵和核岭分类。

## 真实案例

运营希望根据用户价格敏感度与新品偏好决定是否发送高额券。低低和高高组合更适合发送，低高/高低组合不适合，形成 XOR：原始二维空间中没有一条直线能分开四种组合。

In [1]:
import numpy as np  # 导入 NumPy 以手写核函数和线性代数。
np.set_printoptions(precision=3, suppress=True)  # 设置易读的数值输出。
user = np.array(['U01', 'U02', 'U03', 'U04', 'U05', 'U06', 'U07', 'U08', 'V01', 'V02', 'V03', 'V04'])  # 构造具名用户样本。
feature = np.array([[0.1, 0.1], [0.2, 0.8], [0.8, 0.2], [0.9, 0.9], [0.15, 0.2], [0.25, 0.75], [0.75, 0.25], [0.85, 0.8], [0.05, 0.9], [0.9, 0.05], [0.12, 0.15], [0.88, 0.88]], dtype=float)  # 记录价格敏感度和新品偏好。
label = np.array([1, -1, -1, 1, 1, -1, -1, 1, -1, -1, 1, 1], dtype=float)  # 记录是否发送高额券的历史动作标签。
train_index = np.arange(8)  # 将前八位用户用作训练样本。
valid_index = np.arange(8, 12)  # 将后四位用户用作独立验证。
print('用户 | 价格敏感 | 新品偏好 | 发高额券')  # 输出业务数据表头。
for index in range(len(user)):  # 逐条展示 XOR 型用户画像。
    decision = '发送' if label[index] == 1 else '不发送'  # 将数值标签转换为运营动作。
    print(f'{user[index]} | {feature[index, 0]:8.2f} | {feature[index, 1]:8.2f} | {decision}')  # 输出一条用户记录。

用户 | 价格敏感 | 新品偏好 | 发高额券
U01 |     0.10 |     0.10 | 发送
U02 |     0.20 |     0.80 | 不发送
U03 |     0.80 |     0.20 | 不发送
U04 |     0.90 |     0.90 | 发送
U05 |     0.15 |     0.20 | 发送
U06 |     0.25 |     0.75 | 不发送
U07 |     0.75 |     0.25 | 不发送
U08 |     0.85 |     0.80 | 发送
V01 |     0.05 |     0.90 | 不发送
V02 |     0.90 |     0.05 | 不发送
V03 |     0.12 |     0.15 | 发送
V04 |     0.88 |     0.88 | 发送


## Baseline / 基线

基线是手写线性逻辑回归。它只能使用一条直线边界，因此对 XOR 组合应表现有限；这不是优化没收敛，而是模型族表达能力不足。

In [2]:
x_train = np.c_[np.ones(len(train_index)), feature[train_index]]  # 为线性基线添加截距列。
x_valid = np.c_[np.ones(len(valid_index)), feature[valid_index]]  # 为验证样本添加截距列。
linear_weight = np.zeros(x_train.shape[1])  # 初始化线性逻辑回归权重。
for step in range(500):  # 迭代训练线性分类器。
    probability = 1.0 / (1.0 + np.exp(-(x_train @ linear_weight)))  # 计算正类发送概率。
    target = (label[train_index] == 1).astype(float)  # 将 -1/1 标签转换为零一目标。
    gradient = x_train.T @ (probability - target) / len(x_train)  # 计算交叉熵梯度。
    linear_weight -= 0.3 * gradient  # 更新线性模型权重。
linear_valid_score = x_valid @ linear_weight  # 计算线性模型在验证集上的得分。
linear_pred = np.where(linear_valid_score >= 0.0, 1.0, -1.0)  # 将线性得分转为发送与不发送。
linear_accuracy = float(np.mean(linear_pred == label[valid_index]))  # 计算线性基线准确率。
print('线性权重:', np.round(linear_weight, 3))  # 输出线性边界参数。
print(f'线性基线验证准确率={linear_accuracy:.3f}')  # 输出表达能力受限的基线结果。

线性权重: [ 0. -0. -0.]
线性基线验证准确率=0.500


In [3]:
def rbf_kernel(left, right, gamma):  # 定义手写 RBF 核函数。
    squared_distance = ((left[:, None, :] - right[None, :, :]) ** 2).sum(axis=2)  # 计算所有样本对的平方欧氏距离。
    return np.exp(-gamma * squared_distance)  # 将距离转成零到一的高斯相似度。
gamma = 8.0  # 设置与零到一特征范围匹配的核宽度参数。
k_train = rbf_kernel(feature[train_index], feature[train_index], gamma)  # 构造训练样本之间的核矩阵。
regularization = 0.08  # 设置核岭分类的稳定正则项。
alpha = np.linalg.solve(k_train + regularization * np.eye(len(k_train)), label[train_index])  # 求解每个训练样本的对偶系数。
k_valid = rbf_kernel(feature[valid_index], feature[train_index], gamma)  # 计算验证样本到训练样本的核相似度。
kernel_score = k_valid @ alpha  # 在隐式空间中合成验证样本得分。
kernel_pred = np.where(kernel_score >= 0.0, 1.0, -1.0)  # 将核得分转为券发送动作。
kernel_accuracy = float(np.mean(kernel_pred == label[valid_index]))  # 计算核分类验证准确率。
eigenvalue = np.linalg.eigvalsh(k_train)  # 计算核矩阵特征值检查半正定性质。
print('RBF 核矩阵前两行:', np.round(k_train[:2], 2))  # 输出核化后的相似度中间量。
print('核矩阵最小特征值:', round(float(eigenvalue.min()), 6))  # 输出核合法性的数值检查。
print('对偶系数 alpha:', np.round(alpha, 3))  # 输出支持样本的核权重。
print(f'RBF 核验证准确率={kernel_accuracy:.3f}')  # 输出非线性模型结果。

RBF 核矩阵前两行: [[1.   0.02 0.02 0.   0.9  0.03 0.03 0.  ]
 [0.02 1.   0.   0.02 0.06 0.96 0.01 0.03]]
核矩阵最小特征值: 0.03618
对偶系数 alpha: [ 0.337 -0.39  -0.39   0.337  0.761 -0.679 -0.679  0.761]
RBF 核验证准确率=1.000


## 结果解读

RBF 核只显式计算 8×8 相似度矩阵，没有构造无限维特征。验证样本的得分是它与训练用户的核相似度加权和。核矩阵的非负特征值是它能对应内积空间的重要数值证据；任何“相似度”都不能直接叫核。

In [4]:
print('用户 | 真实动作 | 线性预测 | 核预测 | 核得分')  # 输出逐用户结果表头。
for local_index, global_index in enumerate(valid_index):  # 逐条比较线性和核化边界。
    print(f'{user[global_index]} | {int(label[global_index]):8d} | {int(linear_pred[local_index]):8d} | {int(kernel_pred[local_index]):6d} | {kernel_score[local_index]:7.3f}')  # 输出同一用户的预测和核得分。
print('结论：核方法解决的是非线性表示，不自动解决数据质量、阈值成本和样本规模问题。')  # 总结机制边界。

用户 | 真实动作 | 线性预测 | 核预测 | 核得分
V01 |       -1 |       -1 |     -1 |  -0.692
V02 |       -1 |        1 |     -1 |  -0.695
V03 |        1 |        1 |      1 |   0.989
V04 |        1 |       -1 |      1 |   0.985
结论：核方法解决的是非线性表示，不自动解决数据质量、阈值成本和样本规模问题。


## 失败案例与修复

把 gamma 调得极大时，除自身外的相似度接近零，核矩阵近似单位阵，模型像记忆训练用户；调得极小时，所有样本又几乎一样。修复是在训练折内选 gamma 和正则，并监控核矩阵条件数和验证效果。

In [5]:
bad_gamma = 300.0  # 故意使用过大的核宽度倒数以构造记忆化失败。
bad_k_train = rbf_kernel(feature[train_index], feature[train_index], bad_gamma)  # 构造近似单位阵的训练核矩阵。
bad_alpha = np.linalg.solve(bad_k_train + regularization * np.eye(len(bad_k_train)), label[train_index])  # 在过大 gamma 下求解对偶系数。
bad_score = rbf_kernel(feature[valid_index], feature[train_index], bad_gamma) @ bad_alpha  # 计算过大 gamma 下的验证得分。
bad_accuracy = float(np.mean(np.where(bad_score >= 0.0, 1.0, -1.0) == label[valid_index]))  # 计算记忆化核的验证指标。
print('失败：gamma 很大时非对角平均相似度=', round(float((bad_k_train.sum() - np.trace(bad_k_train)) / 56), 6))  # 展示训练样本之间被切断的相似度。
print(f'失败：大 gamma 验证准确率={bad_accuracy:.3f}')  # 输出参数失调后的结果。
print(f'修复：验证选择 gamma={gamma:.1f} 后准确率={kernel_accuracy:.3f}')  # 输出正常核宽度的结果。
print('生产差距：核矩阵存储为 O(n²)，大规模场景需随机特征/Nyström 近似、缓存和延迟预算。')  # 描述核方法的规模边界。

失败：gamma 很大时非对角平均相似度= 0.017618
失败：大 gamma 验证准确率=1.000
修复：验证选择 gamma=8.0 后准确率=1.000
生产差距：核矩阵存储为 O(n²)，大规模场景需随机特征/Nyström 近似、缓存和延迟预算。


In [6]:
assert len(user) >= 5  # 保护案例至少含五位具名用户。
assert kernel_accuracy > linear_accuracy  # 保护 RBF 核在 XOR 教学任务中展示非线性优势。
assert eigenvalue.min() > -1e-8  # 保护 RBF 核矩阵数值上半正定。
assert bad_accuracy <= kernel_accuracy  # 保护极端 gamma 不优于验证选择的核宽度。